# Phytospectra — fine-tune gate: adapter training template

**STATUS: TEMPLATE, NOT RUN.** Prepared per the study prompt; execute only after the fine-tune gate in `FINETUNE-DECISION-GATE.md` opens.

Requirements before running:
1. Pilot benchmark error pattern justifies fine-tuning (gate memo §1).
2. Training data uploaded separately, rights-cleared, expert-reviewed, deduplicated (gate memo §2). This notebook NEVER ships data; it reads a local JSONL of `{messages:[...]}` chat-format examples.
3. Baseline model selected: CHEMLLM-2b (dry run) or ChemLLM-7B-Chat (serious run).

Hardware note (Colab FAQ: https://research.google.com/colaboratory/faq.html): GPU model is not guaranteed. The cell below prints what you actually got and gates the run against it.

In [ ]:
#@title 0. Environment probe — print assigned hardware, install deps
!nvidia-smi || echo 'NO GPU ASSIGNED — request one via Runtime > Change runtime type before continuing'
import torch, json
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    VRAM_GB = p.total_memory / 1e9
    print(f'GPU: {p.name}, {VRAM_GB:.1f} GB VRAM')
else:
    VRAM_GB = 0.0
!pip -q install peft transformers datasets bitsandbytes accelerate 2>&1 | tail -1

In [ ]:
#@title 1. Config — edit these, then run cells in order
BASE_MODEL = "AI4Chem/CHEMLLM-2b-1_5"   # or AI4Chem/ChemLLM-7B-Chat
TRAIN_JSONL = "/content/train.jsonl"    # uploaded separately; rights-cleared
EVAL_JSONL  = "/content/dev.jsonl"      # dev partition only, never test
OUTPUT_DIR  = "/content/drive/MyDrive/phyto_adapter"  # Drive for resume across restarts
MAX_STEPS, BATCH, GRAD_ACC = 500, 2, 8
SEED = 20261102
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
#@title 2. One-batch memory check — REFUSES the full run if it OOMs
# Load tokenizer + quantized model, run ONE forward+backward on ONE example.
# If this cell raises CUDA OOM, do NOT proceed: reduce sequence length or switch
# to a smaller base model. Record the assigned GPU in the run log.
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
        device_map="auto", trust_remote_code=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj"])
model = get_peft_model(model, lora); model.print_trainable_parameters()
one = json.loads(open(TRAIN_JSONL).readline())
batch = tok.apply_chat_template(one["messages"], return_tensors="pt", truncation=True, max_length=2048).to(model.device)
loss = model(input_ids=batch, labels=batch).loss
loss.backward()
print(f'one-batch check OK: loss={loss.item():.3f}; peak mem={torch.cuda.max_memory_allocated()/1e9:.2f} GB / {VRAM_GB:.1f} GB')
model.train()

In [ ]:
#@title 3. Train with checkpoints (restartable)
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
import json
ds = load_dataset("json", data_files={"train": TRAIN_JSONL, "dev": EVAL_JSONL})
def fmt(ex): return tok.apply_chat_template(ex["messages"], tokenize=True, truncation=True, max_length=2048)
ds = ds.map(fmt)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, max_steps=MAX_STEPS, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, lr_scheduler_type="cosine", learning_rate=2e-4,
    bf16=True, logging_steps=20, save_steps=100, save_total_limit=3, seed=SEED,
    report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["dev"],
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train(resume_from_checkpoint=True)  # no-op on first run, resumes after a Colab disconnect

In [ ]:
#@title 4. Export adapter + run log
trainer.save_model(OUTPUT_DIR + "/adapter")
tok.save_pretrained(OUTPUT_DIR + "/adapter")
import json, datetime
log = {"base_model": BASE_MODEL, "steps": MAX_STEPS, "seed": SEED,
       "gpu": torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else "none",
       "finished_utc": datetime.datetime.utcnow().isoformat()}
json.dump(log, open(OUTPUT_DIR + "/run_log.json", "w"), indent=1)
print(json.dumps(log, indent=1))
# Next step happens OFF Colab: evaluate tuned vs untuned on the UNTOUCHED test
# partition with benchmark/blinded_pilot/score_run.py — identical prompts, same seed.